  # II. DESCRIPTIVE ANALYSIS

## 1. Introduction

The objective of this analysis is to identify meaningful patterns, relationships,
and variations within the hospitalized heart-failure patient population.

The analysis focuses on patient characteristics, clinical severity, cardiac
function, laboratory measurements, complications, medications, and
hospitalization outcomes. Rather than only presenting summary statistics,
specific analytical questions will be used to investigate important patterns
within the data.

Each question will be supported by an appropriate analysis and visualization,
followed by an interpretation of the findings. The findings will also help
identify areas that can be explored further through prescriptive and predictive
analysis.

### 2. Import Libraries

The analysis uses Pandas and NumPy for data manipulation and numerical
analysis, while Matplotlib and Seaborn are used to create clear and
informative visualizations.

In [6]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Visualization style
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


### 3. Load Master Patient Dataset

The cleaned patient-level datasets were combined during the data preparation
stage to create the master patient table. This table contains one row per
patient and serves as the common analytical dataset for the descriptive,
prescriptive, and predictive stages.

The master table is loaded from the cleaned-data folder so that this notebook
uses the same validated analytical population without repeating the earlier
cleaning and merging process.

In [ ]:
# Use the team's original data folder when it is available.
TEAM_DATA_PATH = Path(
    r"C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026"
)

# Jupyter normally starts in the notebook folder; also check its parent.
# In either folder, the cleaning notebook should have created cleaned_data/patient_master.csv.
candidate_folders = [TEAM_DATA_PATH, Path.cwd(), Path.cwd().parent]
DATA_PATH = next(
    (folder for folder in candidate_folders
     if (folder / "cleaned_data" / "patient_master.csv").is_file()),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Cannot find cleaned_data/patient_master.csv. Run the team's cleaning "
        "notebook first, or open this notebook from the project folder."
    )

CLEANED_PATH = DATA_PATH / "cleaned_data"
MASTER_PATH = CLEANED_PATH / "patient_master.csv"
patient_master = pd.read_csv(MASTER_PATH)

print("Master patient dataset loaded from:", MASTER_PATH)
print(f"Rows: {patient_master.shape[0]:,}")
print(f"Columns: {patient_master.shape[1]:,}")


In [9]:
# Create the dataframe used for descriptive analysis

df = patient_master.copy()

print("Descriptive analysis dataframe created.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Descriptive analysis dataframe created.
Rows: 2,008
Columns: 198


### 4. Initial Dataset Validation

Before beginning the descriptive questions, the master dataset is checked
to confirm that the expected patient-level structure has been preserved.

The checks focus on the dataset size, patient identifier uniqueness,
duplicate records, data types, and missing values. This provides confidence
that the analysis is being performed on the intended analytical population
and helps identify variables that may require consideration when answering
individual questions.

In [10]:
# Dataset structure

print("Dataset Shape:", df.shape)

print("\nNumber of Patients:", df["inpatient_number"].nunique())

print("Duplicate Patient IDs:",
      df["inpatient_number"].duplicated().sum())

print("Duplicate Rows:",
      df.duplicated().sum())

Dataset Shape: (2008, 198)

Number of Patients: 2008
Duplicate Patient IDs: 0
Duplicate Rows: 0


In [11]:
# Data type summary

print("Data Type Summary:")
display(df.dtypes.value_counts())

Data Type Summary:


float64    149
int64       30
object      14
bool         5
Name: count, dtype: int64

In [12]:
# Missing-value overview

missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

missing_summary = missing_summary[missing_summary > 0]

print("Columns with missing values:", len(missing_summary))

display(missing_summary.head(30))

Columns with missing values: 123


respiratory_support                             1966
time_of_death_days_from_admission               1964
homocysteine                                    1862
apolipoprotein_a                                1832
lipoprotein                                     1832
apolipoprotein_b                                1832
tricuspid_valve_return_pressure                 1826
erythrocyte_sedimentation_rate                  1701
ea                                              1615
myoglobin                                       1610
serum_magnesium                                 1601
inorganic_phosphorus                            1601
mitral_valve_ams                                1458
glutamic_oxaliplatin                            1416
lvef                                            1373
tricuspid_valve_return_velocity                 1218
time_to_emergency_department_within_6_months    1111
readmission_time_days_from_admission            1107
high_sensitivity_protein                      

## Question 1 — How many patients are in each NYHA class and Killip grade?

**Why these fields?** NYHA class describes limits on physical activity from heart failure symptoms. Killip grade records clinical signs of cardiac severity. Together, their *separate* distributions describe this patient group at admission. They measure different things, so matching class numbers would not mean the two assessments agree.

**How we answer it:** Check that both fields contain the grades documented in the team's cleaned table, count patients in each grade, and calculate each grade's share of all patients. Show missing records separately so the percentages have a clear denominator. Then plot the two distributions. A later question can examine how the two grades overlap within the same patients.

The [American Heart Association explains NYHA classes](https://www.heart.org/en/health-topics/heart-failure/what-is-heart-failure/classes-of-heart-failure); the dataset's data dictionary defines the recorded Killip grades.


In [ ]:
# Step 1: Identify the two cleaned columns and the grades we expect in each.
severity_fields = {
    "NYHA class": ("nyha_cardiac_function_classification", [2, 3, 4]),
    "Killip grade": ("killip_grade", [1, 2, 3, 4]),
}

# Check the column names before calculating anything. A missing column means
# the wrong version of patient_master.csv may have been loaded.
missing_columns = [column for column, _ in severity_fields.values() if column not in df.columns]
if missing_columns:
    raise KeyError(f"Severity columns missing from patient_master.csv: {missing_columns}")

severity_tables = {}
for name, (column, possible_grades) in severity_fields.items():
    # Convert to numbers, but stop if a non-empty value is not a documented grade.
    numeric = pd.to_numeric(df[column], errors="coerce")
    invalid = df[column].notna() & (numeric.isna() | ~numeric.isin(possible_grades))
    if invalid.any():
        raise ValueError(f"{name} has {int(invalid.sum())} unrecognized grade(s); review the source.")

    # Include grades with zero patients to make the distributions easy to compare.
    counts = numeric.value_counts().reindex(possible_grades, fill_value=0).astype(int)
    severity_tables[name] = pd.DataFrame({
        "Grade": possible_grades,
        "Patients": counts.to_numpy(),
        "Percent of all patients": (counts.to_numpy() / len(df) * 100).round(1),
    })

    # The denominator is the full master table; show missing grades explicitly.
    print(f"\n{name}: {int(numeric.notna().sum()):,} recorded; {int(numeric.isna().sum()):,} missing")
    display(severity_tables[name])


In [ ]:
# Step 2: Show one bar chart per clinical assessment, with both counts and percentages.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, summary) in zip(axes, severity_tables.items()):
    bars = ax.bar(summary["Grade"].astype(str), summary["Patients"], color="#3274a1")
    ax.set(title=f"Patients by {name}", xlabel=name, ylabel="Number of patients")
    ax.set_ylim(0, max(1, summary["Patients"].max()) * 1.16)
    for bar, count, percent in zip(bars, summary["Patients"], summary["Percent of all patients"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{count:,} ({percent:.1f}%)", ha="center", va="bottom", fontsize=9)

fig.suptitle("Admission severity: two different clinical assessments", fontsize=14)
plt.tight_layout()
plt.show()

# State a finding using the real counts after the dataset has been loaded.
for name, summary in severity_tables.items():
    largest = summary.loc[summary["Patients"].idxmax()]
    print(f"Most common {name}: {int(largest['Grade'])} — "
          f"{int(largest['Patients']):,} patients ({largest['Percent of all patients']:.1f}% of all patients).")


### What Question 1 tells us

The tables and charts establish how common each recorded severity grade is in the full patient group. Use the displayed percentages and missing counts when describing the result. A large count for one grade tells us that grade was common *in this dataset*; it does not explain what caused the patient's condition.


## Question 2 — How do NYHA classes and Killip grades overlap in the same patients?

**Why this question?** Question 1 counts each grade separately. A table of patients who have *both* recorded scores shows which combinations occurred together, including whether patients within one NYHA class received several different Killip grades. This helps the team decide what comparisons to explore later.

**How we answer it:** Keep only patients with both recorded grades. Cross-tabulate their NYHA class and Killip grade, then divide each NYHA row by its own total. A heatmap displays the counts and within-class percentages. We also report how many patients were left out for missing grades.

**Interpretation limit:** NYHA and Killip assess different aspects of illness. Their numerical grade labels do not mean the same thing, so a diagonal on the chart is **not** an agreement rate. Formal correlation and outcome comparisons belong in the separate prescriptive notebook.


In [ ]:
# Step 1: Use the same patient-level table as Question 1.
nyha_column = "nyha_cardiac_function_classification"
killip_column = "killip_grade"
severity_pairs = df[[nyha_column, killip_column]].dropna().copy()

# Count every observed NYHA/Killip combination; keep the expected grade order.
pair_counts = pd.crosstab(
    severity_pairs[nyha_column].astype(int),
    severity_pairs[killip_column].astype(int),
).reindex(index=[2, 3, 4], columns=[1, 2, 3, 4], fill_value=0)
pair_counts.index.name = "NYHA class"
pair_counts.columns.name = "Killip grade"

# Each percentage is within one NYHA class (each nonempty row sums to 100%).
row_totals = pair_counts.sum(axis=1)
pair_percent = pair_counts.div(row_totals.replace(0, np.nan), axis=0).mul(100).round(1)

print(f"Patients with both grades: {len(severity_pairs):,} of {len(df):,}")
print(f"Patients missing one or both grades: {len(df) - len(severity_pairs):,}")
print("\nNumber of patients in each combination:")
display(pair_counts)
print("Percent within each NYHA class (rows):")
display(pair_percent)


In [ ]:
# Step 2: Label each heatmap cell with both the patient count and the row percentage.
labels = np.array([
    [f"{pair_counts.loc[nyha, killip]:,}\n{pair_percent.loc[nyha, killip]:.1f}%"
     if row_totals.loc[nyha] else "No patients"
     for killip in pair_counts.columns]
    for nyha in pair_counts.index
])

plt.figure(figsize=(10, 5))
sns.heatmap(
    pair_percent.fillna(0), annot=labels, fmt="", cmap="Blues", vmin=0, vmax=100,
    cbar_kws={"label": "Percent within NYHA class"}, linewidths=0.5,
)
plt.title("How NYHA classes and Killip grades overlap")
plt.xlabel("Killip grade")
plt.ylabel("NYHA class")
plt.tight_layout()
plt.show()

# Give the reader a factual result without claiming that matching numbers mean agreement.
largest_pair = pair_counts.stack().idxmax()
largest_count = int(pair_counts.loc[largest_pair])
print(f"Most frequent combination: NYHA {largest_pair[0]} and Killip {largest_pair[1]} "
      f"({largest_count:,} patients, {largest_count / len(severity_pairs):.1%} of patients with both grades).")


### What Question 2 tells us

Read *across a row* to see how Killip grades are distributed among patients in one NYHA class. The percentage denominator changes from row to row and is printed in the table above. The chart describes overlap in this patient group; it cannot tell us that one score causes the other or that the two assessments measure the same condition.
